<a href="https://colab.research.google.com/github/Amarildo-CCNA/Analise-de-Dados-Python-SQL/blob/main/Miniprojeto_Amarildo_Xavier_Marchi_Analise_de_Dados_T1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Mini Projeto - Análise de Dados com Python**

---



# Sprint 1 - Importação dos dados e Bibliotecas


* Montagem do Drive onde está localizada a base de dados
* Importar as bibliotecas básicas para;
  * Leitura dos Dados
  * Tratamento das Anomalias
  * Análise dos Insights
* Importar o arquivo Base_Varejo.csv
* Visualização básica da base de dados



### Importação de Bibliotecas Necessárias e Montagem do Drive.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

### Efetuando a Carga da Base de Dados e leitura básica.

In [ ]:
file_path = '/content/drive/MyDrive/Mini_Projeto/Base_de_Dados/Base_Varejo.csv'
df_raw = pd.read_csv(file_path, sep=';')
display(df_raw.head())


### Problemas Básicos Identificados;


* A base de dados é .csv. Mas, veio com separador ponto e vírgula.
* Colunas sem dados ou inconsistências.
* Registros duplicados, 96.131 linhas.



In [ ]:
df_raw.info()

In [ ]:
df_raw.shape

# Sprint 2 - Transformação dos Dados

In [ ]:
def formata_data(df_in):
    """Padronização das datas, substituindo separadores de forma vetorizada."""
    df_out = df_in.copy()
    # str.replace vetorizado substitui '-' ou '.' por '/' instantaneamente em C
    df_out['DATA'] = df_out['DATA'].astype(str).str.strip().str.replace(r'[-.]', '/', regex=True)
    return df_out

def capitaliza_strings(df_in):
    """Remoção dos espaços e capitalização das palavras de forma vetorizada (Sem .apply)."""
    df_out = df_in.copy()
    colunas_texto = ['CL_GENERO', 'CL_SEG', 'PR_CAT', 'PR_NOME']

    # .str.title() processa a string inteira e capitaliza após qualquer borda/espaço
    # .str.strip() remove espaços nas extremidades de forma ultra rápida
    df_out[colunas_texto] = df_out[colunas_texto].astype(str).stack().str.strip().str.title().unstack()
    return df_out

def limpa_inteiros(df_in):
    """Converção dos identificadores para inteiros usando substituição vetorizada rápida."""
    df_out = df_in.copy()
    colunas_int = ['CO_ID', 'CL_ID', 'CL_EC', 'CL_FHL', 'PR_ID']

    # Aplica a limpeza de caracteres não numéricos em lote
    df_out[colunas_int] = df_out[colunas_int].astype(str).stack().str.replace(r'\D', '', regex=True).unstack()

    # Converte os tipos em bloco de uma só vez
    df_out[colunas_int] = df_out[colunas_int].apply(pd.to_numeric, errors='coerce').astype('Int64')
    return df_out

# ==========================================
#    EXECUÇÃO DO PIPELINE COM OTIMIZAÇÔES
# ==========================================

df = (df_raw
            .pipe(formata_data)
            .pipe(capitaliza_strings)
            .pipe(limpa_inteiros))

display(df.head())

In [ ]:
df.info()

# Sprint 3 - Limpeza de Nulos e Duplicatas

In [ ]:
# Eliminando as linhas duplicadas e mantendo a primeira ocorrência
df_limpo = df.drop_duplicates(ignore_index=False, keep='first').drop(columns=['Unnamed: 10','Unnamed: 11','Unnamed: 12','Unnamed: 13'], errors='ignore')

# Converter colunas de texto para o tipo datetime
# O argumento 'errors='coerce'' transforma datas inválidas em NaT (nulo)
df_limpo['DATA'] = pd.to_datetime(df_limpo['DATA'], errors='coerce', dayfirst=True)

# Exibir resultado final
display(df_limpo.head())

In [ ]:
print(f'Linhas duplicadas: {df_limpo.duplicated().sum()}')

### Justificativa para a supressão de linhas duplicadas

*   Das 829.999 linhas de dados, 96.131 são repetidas. Isso representa um percentual de 11,58 % das linhas. Eu considerei o impacto da retirada dessas linhas menor do que o viés estatístico de deixá-las no Dataframe.




In [ ]:
df_limpo.info()

# Sprint 4 - Estatística Descritiva

In [ ]:
# estatística descritiva básica para coluna de número de filhos do cliente

col_fhl = df_limpo['CL_FHL']

estatisticas = {
    'Média': col_fhl.mean(),
    'Mediana': col_fhl.median(),
    'Desvio Padrão': col_fhl.std(),
    'Moda': col_fhl.mode().tolist(),  # Lista os valores mais frequentes
    'Mínimo': col_fhl.min(),
    'Máximo': col_fhl.max(),
    'Contagem': col_fhl.count(),
    '1º Quartil (Q1)': col_fhl.quantile(0.25),
    '3º Quartil (Q3)': col_fhl.quantile(0.75)
}

# Exibe o relatório formatado
for metrica, valor in estatisticas.items():
    print(f'{metrica}: {valor}')

## Número de compras por número de filhos do cliente

In [ ]:
media_filhos = pd.DataFrame(df_limpo.groupby(['CL_FHL'])['PR_ID'].count()).reset_index()
g = sns.barplot(
    data=media_filhos,
    x='CL_FHL',
    y='PR_ID',
    hue='CL_FHL',
    palette='viridis',
    legend=False
)
g.set_ylabel('Contagem de compras')
g.set_xlabel('Número de Filhos')

In [ ]:
produtos_mais_vendidos = pd.DataFrame(df_limpo.groupby(['PR_CAT','PR_NOME'])['PR_ID'].count().reset_index())
produtos_mais_vendidos = produtos_mais_vendidos[produtos_mais_vendidos['PR_CAT'] != '#N/D']
produtos_mais_vendidos = produtos_mais_vendidos.rename(columns={'PR_ID':'Contagem','PR_CAT':'Categoria','PR_NOME':'Nome do produto'})
produtos_mais_vendidos = produtos_mais_vendidos.groupby('Categoria').apply(lambda x: x.nlargest(1,'Contagem'))

produtos_mais_vendidos = produtos_mais_vendidos.drop(columns=['Categoria']).reset_index()
sns.barplot(data=produtos_mais_vendidos,x='Nome do produto',y='Contagem', hue='Nome do produto', palette='viridis', legend=False)
plt.xticks(rotation=45)
plt.title('Maior ocorrencia de vendas em cada categoria')

# Sprint 5 - Relatório e Documentação

## Conclusões baseadas nas estatísticas e no gráfico


* Pessoas sem filhos compraram mais do que pessoas com filhos.
* Isso pode indicar que o mercado situa-se em área universitária.
* Também pode inicar que o mercado venda produtos mais básicos.
* O produto mais vendido é o presunto cozido.
* O número de crianças na vizinhança é semelhante ao número de pets.

## Possíveis problemas remanescentes na base de dados


*   Não é possível ter uma visão de correlação entre os dados.


